In [ ]:
# Geochem aggregation to grid cells
import pandas as pd
import numpy as np
import geopandas as gpd

# --- 1. Load Phase 1 outputs ---
geochem = pd.read_csv("../data/processed/geochem_clean.csv")
grid    = gpd.read_file("../data/processed/grid_labels.geojson").to_crs("EPSG:32612")

# Sanity: confirm the threshold columns from Phase 1 survived the CSV round-trip
required = ["Mo_above_thresh", "As_above_thresh", "Au_above_thresh",
            "Cu_imputed", "Pb_imputed"]
missing = [c for c in required if c not in geochem.columns]
assert not missing, f"Missing columns from Phase 1: {missing}"



In [ ]:
# --- 2. Sample points → GeoDataFrame in analysis CRS ---
geochem_gdf = gpd.GeoDataFrame(
    geochem,
    geometry=gpd.points_from_xy(geochem["lon"], geochem["lat"]),
    crs="EPSG:4326",
).to_crs("EPSG:32612")



In [ ]:
# --- 3. Spatial join: assign cell_id to every sample ---
samples_with_cell = gpd.sjoin(
    geochem_gdf,
    grid[["cell_id", "deposit", "geometry"]],
    how="inner",           # drops the 2 NM-border samples automatically
    predicate="within",
)
print(f"Samples in cells: {len(samples_with_cell)} / {len(geochem_gdf)}")


In [ ]:

# --- 4. Convert pd.NA → float so .mean() returns fraction, not error ---
# True→1.0, False→0.0, NA→NaN. .mean() then averages over non-NaN.
for el in ["Mo", "As", "Au"]:
    samples_with_cell[f"{el}_above_thresh"] = (
        samples_with_cell[f"{el}_above_thresh"]
        .astype("boolean")     # handles pd.NA
        .astype(float)         # then → float with NaN for NA
    )



In [ ]:
# --- 5. Named aggregation ---
agg_spec = {}
for el in ["Cu", "Pb"]:
    agg_spec[f"{el}_mean"] = (f"{el}_imputed", "mean")
    agg_spec[f"{el}_max"]  = (f"{el}_imputed", "max")

for el in ["Mo", "As", "Au"]:
    agg_spec[f"{el}_frac_above"]   = (f"{el}_above_thresh", "mean")
    agg_spec[f"{el}_frac_unknown"] = (f"{el}_above_thresh",
        lambda s: float(s.isna().mean()))

# Diagnostics — NOT features. Kept for Phase 4 experiments.
agg_spec["n_samples"] = ("cell_id", "size")

cell_geochem = (
    samples_with_cell
    .groupby("cell_id")
    .agg(**agg_spec)
    .reset_index()
)
print(f"Cells with ≥1 sample: {len(cell_geochem)}")


In [ ]:

# --- 6. Join to grid; keep all 3,064 cells (left join) ---
grid_features = (
    grid[["cell_id", "deposit", "geometry"]]
    .merge(cell_geochem, on="cell_id", how="left")
)

# Diagnostics
grid_features["has_geochem"] = grid_features["n_samples"].notna()
grid_features["n_samples"]   = grid_features["n_samples"].fillna(0).astype(int)



In [ ]:
# --- 7. Report ---
print(f"\nGrid frame shape: {grid_features.shape}")
print(f"Cells with any geochem: {grid_features['has_geochem'].sum()} / {len(grid_features)}")
print(f"\nColumn list:")
print(grid_features.drop(columns=["geometry"]).columns.tolist())
print(f"\nNon-null counts per feature:")
print(grid_features.drop(columns=["geometry"]).notna().sum())